## Scraping Agents Workflow Architecture Prototyping 

The main agents that will be tested are going to be: 
- Item Selection Agent: chooses which listings may be valid and worth investigating, selects like 3-9 listings, and then 1-3 are chosen and the rest are used as backup in case the listings researched were invalid.
- Orchestrator Agent: When given a listing deemed worthy of investigating, coordinates and monitors spawned sub-agents, collects their findings and flagged ambiguities, and delivers a final buy/watch/ignore verdict with reasoning.
- eBay Sub-Agent: Searches eBay for sold listings matching the given item, normalizes results against the source listing (title, condition, colorway, year where available), flags ambiguous matches, and returns structured comp data to the orchestrator.

## Agent-To-Agent-Interactions 
Item Selection <--> Item Selection: 
- only interacts with itself, and once the listings are found, the data is stored and gone through sequentially, spawning one orchestrator agent at at time. 

Orchestrator -> eBay: 
- resolves eBay listing confusion
- monitors sub-agents(can pause, stop, or kill a sub-agent) 

eBay -> Orchestrator: 
- flags eBay listing and calls Orchestrator agent to help resolve confusion

# Agent Inputs 
Item Selection: 
- structured/parsed html from TheRealReal brand catalog page 

Orchestrator: 
- all info pertaining to a given listing, use schemas/models based on the category of the garment 
- eBay flagged listing info

eBay: 
- all info pertaining to a given listing, use schemas/models based on the category of the garment 

# Agent Outputs
- TODO

# Important Data Centralizations
- All item listing info is centralized into one structure that ALL agents will use to make context injection easier. 
- All listings regardless if they are found on eBay or TheRealReal all share the same model/schema based on the type of garment(shoes, tshirt, jacket, etc). 
- 


In [ ]:
from typing import TypedDict, Annotated                                                                          
  from langgraph.graph import StateGraph                                                                           
  import operator                                                                                                  
                                                                                                                   
  class AgentState(TypedDict):                                                                                     
      listing: dict                                                                                                
      ebay_comps: list                                                                                           
      flags: Annotated[list, operator.add]  # agents append, don't overwrite
      verdict: str 

In [ ]:
 def orchestrator_node(state: AgentState):
      # reason over state, return partial update
      return {"verdict": "buy"}

  def ebay_agent_node(state: AgentState):
      return {"ebay_comps": [...], "flags": ["ambiguous colorway"]}

# Listing Selection Agent Prototype
Processes TRR listings in batches of 2, accumulates picks until quota (2) is met, then stops. Once listings are found, it may be stored either in state and/or SQLite, still TBD. Doesn't include how html is refetched, only works once there is html for a listing catalog that already exists. 

HTML FOLDER ARCHITECTURE(located inside of db/)
/html
/catalog -> vivienne_westwood_2026-06-12.html : brand_name, year, month, day
/listing ->  item_67890.html : item and a basic UUID 

- Discrete Workflow Steps for Listing Selection 

1. get_listings(brand: str) # check dir, pull html download from directory 
2. parse_html(html : str) # get x listings through regex or key-value matching(html already has structure) 
3. classify_listings() # classifies listings to filter what is worth keeping/investigating 

- Non-functional helper components, but will be needed for testing after workflow is completed  
show_selected_listings() 
show_discarded_listings() 
show_confusing_listings() 
- all of these show listing meta-data along with image and reason for classification based on the listings and their received categories

enum ClothingType:  
tshirt : str, 
jacket : str, 
dress: str, 
etc... 

Catalog Individual Listing Schema: 
name : str 
brand : str
listing_url: str 
image_url: str 
classification : {type: str, reason: str} 

** NULLABLE FIELDS ** determined by LLM and/or after a listing is classified as worth researching  
type: ClothingType(enum) # TBD by LLM after picking listings(urls show the type of item)
size : str 
description: str 
condition: str  
** except "type", all other fields are determined after the listing's detailed html is pulled after being deemed worthy to keep researching ** 

get_listings(brand : str) -> list[any]: 
- straightforward, return listings in a list of dicts 

parse_html(html : Object) -> list[any]: 
- return structured dicts based on the listings 

classify_listings(listings : any) -> list[any] 
- static context: 
- dynamic context: item info 
- takes max 3 listings and returns with classification types/reason 



In [ ]:
### HTML Parser — Layer 1: JSON-LD Collection
# Parses the structured data blob embedded in the page — gives name, url, image for 
# all listings in one shot.

from bs4 import BeautifulSoup
import json

catalog_file_name = "clean_catalog.html"

with open(f"{catalog_file_name}", "r") as f:
    soup = BeautifulSoup(f, "lxml")

# --- Step 1: parse the JSON-LD collection ---
script = soup.find("script", {"type": "application/ld+json"})
data = json.loads(script.string)

collection_name = data["name"]           # e.g. "undercover"
collection_url  = data["url"]
listings_raw    = data["mainEntity"]["itemListElement"]

print(f"Collection : {collection_name}")
print(f"URL        : {collection_url}")
print(f"Listings   : {len(listings_raw)}")


In [ ]:
from bs4 import BeautifulSoup
import json

with open(f"{catalog_file_name}", "r") as f:
  html = f.read()

  soup = BeautifulSoup(html, "lxml")

  # Target the JSON-LD script tag
  # script = soup.find("script", {"type": "application/ld+json"})
  # data = json.loads(script.string)
  print(len(html))
  print(html[:500])

  # Navigate to listings
  # listings = data["mainEntity"]["itemListElement"]
  # Each listing: {"@type": "ListItem", "position": 1, "name": "...", "url": "...", "image": "..."}


In [ ]:
import re

cards = soup.find_all("div", {"data-testid": re.compile(r"^plp-product/\d+$")})

def get_text(card, testid_suffix):
    el = card.find(attrs={"data-testid": re.compile(testid_suffix + "$")})
    return el.get_text(strip=True) if el else None

def extract_price_fields(card):
    return {
        "price_original" : get_text(card, "-price-original"),
        "price_final"    : get_text(card, "-price-final"),
        "price_callout"  : get_text(card, "-price-callout"),
    }


In [ ]:
import re

fields = ["name", "listing_url", "image_url", "size", "price_original", "price_final"]
counts = {f: 0 for f in fields}
total = min(len(cards), len(listings_raw))

for i, card in enumerate(cards[:total]):
    meta        = listings_raw[i]
    prices      = extract_price_fields(card)
    product_id  = card["data-testid"].split("/")[1]

    row = {
        "name"           : meta.get("name"),
        "listing_url"    : meta.get("url"),
        "image_url"      : meta.get("image"),
        "size"           : get_text(card, "-size"),
        "price_original" : prices["price_original"],
        "price_final"    : prices["price_final"],
    }

    for f in fields:
        if row.get(f):
            counts[f] += 1

    callout = f" ({prices['price_callout']})" if prices["price_callout"] else ""
    print(f"[{product_id}] {row['name']}")
    print(f"  size     : {row['size']}")
    print(f"  price    : {row['price_original']}")
    print(f"  final    : {row['price_final']}{callout}")
    print(f"  image    : {row['image_url']}")
    print(f"  url      : {row['listing_url']}")
    print()

print("=" * 40)
print(f"Coverage report ({total} listings)\n")
for f in fields:
    pct = (counts[f] / total * 100) if total else 0
    bar = "#" * int(pct / 5)
    print(f"  {f:<18} {pct:5.1f}%  {bar}")

In [ ]:
# Module-level tool definitions. create_agent reads the @tool decorator's name
# + description and injects them into the model's system prompt automatically —
# so the docstrings/decorator strings here are what the LLM actually sees.
from langchain_core.tools import tool

# Fixed defaults for the HTML inspection tools — model picks only the
# testid_suffix. Pinned to keep tool results small and consistent.
CONTEXT_CHARS = 40
CONTEXT_LIMIT = 10
CARD_LIMIT = 10
TESTID_LIMIT = 10

# Tools need to read live HTML, but the @tool decorator wants plain functions.
# ListingSelectionAgent sets this ref before invoking the heal agent. Single-
# instance pattern is fine here — only one heal runs at a time per process.
_html_tools_ref = None


def bind_html_tools(html_tools) -> None:
    """Point the module-level tools at a live HTMLTools instance."""
    global _html_tools_ref
    _html_tools_ref = html_tools


@tool(
    "get_card_testids",
    description=(
        "Return every unique data-testid value found inside one product card. "
        "Use this to discover which testids exist on the page before probing them."
    ),
)
def get_card_testids() -> list[str]:
    return _html_tools_ref.get_card_testids(card_limit=CARD_LIMIT, limit=TESTID_LIMIT)


@tool(
    "get_element_context",
    description=(
        "Return HTML snippets surrounding elements whose data-testid ends with "
        "`testid_suffix`. Use this to confirm a candidate selector points at "
        "the value you expect (e.g. a price string contains '$' or digits)."
    ),
)
def get_element_context(testid_suffix: str) -> list[str]:
    return _html_tools_ref.get_element_context(
        testid_suffix,
        chars=CONTEXT_CHARS,
        card_limit=CARD_LIMIT,
        limit=CONTEXT_LIMIT,
    )


HTML_TOOLS = [get_card_testids, get_element_context]


In [ ]:
# LangSmith tracing — set LANGSMITH_API_KEY + LANGSMITH_PROJECT in .env.
# Must run BEFORE langchain/langgraph imports below so the tracer attaches.
import os
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "trr-healer")

print(f"LANGSMITH_TRACING = {os.environ.get('LANGSMITH_TRACING')}")
print(f"LANGSMITH_PROJECT = {os.environ.get('LANGSMITH_PROJECT')}")
print(f"LANGSMITH_API_KEY set? {bool(os.environ.get('LANGSMITH_API_KEY'))}")


In [ ]:
import os
import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import TypedDict, NotRequired, Optional

from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image, display

# discrete steps for workflow:
# 1. create_catalog_batch  - kicks off CDP fetch via HTMLTools.generate_catalog_html, returns batch_id
# 2. request_catalog       - verifies the HTML landed on disk and resolves its path
# 3. parse_catalog_html    - delegates JSON-LD + DOM parsing to HTMLTools.clean_catalog_html
# 4. calculate_coverage    - % per field; below threshold -> heal
# 5a. heal_dom_selectors   - multi-field heal: full config in / full config out
# 5b. heal_single_field    - single-field heal: scoped suffix in / scoped patch out, merged
# 6. classify_listings     - pops batch of 3, LLM categorizes selected/discarded/confused
# 7. evaluate_picks        - router: quota met -> save, more needed -> loop, exhausted -> ERROR
# 8. save_selected_listings - TODO (deferred until catalog structure decisions are made)

@dataclass
class Listing:
    name: Optional[str] = None
    brand: Optional[str] = None
    listing_url: Optional[str] = None
    image_url: Optional[str] = None
    price: Optional[str] = None
    final_price: Optional[str] = None
    size: Optional[str] = None
    classification: Optional[dict] = None

    @classmethod
    def fields_for_coverage(cls) -> dict[str, float]:
        return {
            "name": 0.75,
            "listing_url": 0.75,
            "image_url": 0.75,
            "size" : 0.0,
            "price": 0.95,
            "final_price": 0.40,
        }

GEMINI_MODEL = "gemini-2.5-flash"


class AgentEvals:
    def __init__(self):
        self.heal_calls = 0
        self.classify_batches = 0
        self.quota_retries = 0
        self.batch_results = []
        self.coverage_improvements = []
        self._pre_heal_coverage = None

    def record_heal_start(self, coverage: dict):
        self.heal_calls += 1
        self._pre_heal_coverage = dict(coverage) if coverage else {}

    def record_post_heal_coverage(self, coverage: dict):
        if self._pre_heal_coverage is None:
            return
        before = self._pre_heal_coverage
        deltas = [coverage.get(f, 0) - before.get(f, 0) for f in before]
        if deltas:
            self.coverage_improvements.append(sum(deltas) / len(deltas))
        self._pre_heal_coverage = None

    def record_classify_batch(self, selected: int, discarded: int, confused: int):
        self.classify_batches += 1
        if self.classify_batches > 1:
            self.quota_retries += 1
        self.batch_results.append({
            "selected": selected, "discarded": discarded, "confused": confused
        })

    def avg_coverage_improvement(self) -> float:
        if not self.coverage_improvements:
            return 0.0
        return sum(self.coverage_improvements) / len(self.coverage_improvements)

    def keep_discard_ratio(self) -> float:
        kept = sum(b["selected"] for b in self.batch_results)
        discarded = sum(b["discarded"] for b in self.batch_results)
        if discarded == 0:
            return float("inf") if kept else 0.0
        return kept / discarded


class ListingSelectionState(TypedDict):
    brand: str
    worthy_listing_quota: NotRequired[int]
    max_classify_batches: NotRequired[int]
    batch_id: NotRequired[str]
    scrape_config: NotRequired[dict]

    listings: NotRequired[list]
    selected_listings: NotRequired[list]
    discarded_listings: NotRequired[list]
    confused_listings: NotRequired[list]

    catalog_html_path: NotRequired[str]
    last_coverage: NotRequired[dict]
    selector_history: NotRequired[list]
    retries: NotRequired[int]
    failed_fields: NotRequired[list]


class ListingSelectionAgent:
    DEFAULT_QUOTA = 3
    DEFAULT_MAX_BATCHES = 5
    COVERAGE_THRESHOLD = 0.75
    HEAL_RETRY_CAP = 3
    HEAL_RECURSION_LIMIT = 8

    def __init__(self, brand: str):
        self.brand = brand
        self.evals = AgentEvals()
        self._html_tools = None
        self._heal_agent = None
        self._classify_llm = None
        api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
        if api_key:
            os.environ["GOOGLE_API_KEY"] = api_key
        self.graph = self.build_graph()

    @property
    def html_tools(self):
        if self._html_tools is None:
            from src.scraper.tools.html import HTMLTools
            self._html_tools = HTMLTools(self.brand)
            # point module-level @tool wrappers at this HTMLTools instance
            bind_html_tools(self._html_tools)
        return self._html_tools

    # Canonical LangChain pattern: build the agent once with HTML_TOOLS bound.
    # create_agent reads each @tool's name + description for the system prompt,
    # so the heal nodes don't need to restate tool docs.
    @property
    def heal_agent(self):
        if self._heal_agent is None:
            # ensure html_tools ref is bound before the agent uses tools
            _ = self.html_tools
            llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0)
            self._heal_agent = create_agent(llm, tools=HTML_TOOLS)
        return self._heal_agent

    @property
    def classify_llm(self):
        if self._classify_llm is None:
            self._classify_llm = ChatGoogleGenerativeAI(
                model=GEMINI_MODEL,
                temperature=0,
                model_kwargs={"response_mime_type": "application/json"},
            )
        return self._classify_llm

    def build_graph(self):
        builder = StateGraph(ListingSelectionState)
        builder.add_node("create_catalog_batch", self.create_catalog_batch)
        builder.add_node("request_catalog", self.request_catalog)
        builder.add_node("parse_catalog_html", self.parse_catalog_html)
        builder.add_node("calculate_coverage", self.calculate_coverage)
        builder.add_node("heal_dom_selectors", self.heal_dom_selectors)
        builder.add_node("heal_single_field", self.heal_single_field)
        builder.add_node("classify_listings", self.classify_listings)
        builder.add_node("save_selected_listings", self.save_selected_listings)

        builder.set_entry_point("create_catalog_batch")
        builder.add_edge("create_catalog_batch", "request_catalog")
        builder.add_edge("request_catalog", "parse_catalog_html")
        builder.add_edge("parse_catalog_html", "calculate_coverage")
        builder.add_conditional_edges("calculate_coverage", self.route_coverage, {
            "HEAL": "heal_dom_selectors",
            "ONE_FAIL_CASE": "heal_single_field",
            "PASS": "classify_listings",
            "ERROR": END,
        })
        builder.add_edge("heal_dom_selectors", "parse_catalog_html")
        builder.add_edge("heal_single_field", "parse_catalog_html")
        builder.add_conditional_edges("classify_listings", self.evaluate_picks, {
            "NOT_ENOUGH": "classify_listings",
            "CONTINUE": "save_selected_listings",
            "ERROR": END,
        })
        builder.add_edge("save_selected_listings", END)

        return builder.compile()

    # ---------- routers ----------

    def route_coverage(self, state: ListingSelectionState) -> str:
        if state.get("retries", 0) >= self.HEAL_RETRY_CAP:
            return "ERROR"
        failed = state.get("failed_fields", [])
        if len(failed) == 1:
            return "ONE_FAIL_CASE"
        if failed:
            return "HEAL"
        return "PASS"

    def evaluate_picks(self, state: ListingSelectionState) -> str:
        quota = state.get("worthy_listing_quota", self.DEFAULT_QUOTA)
        max_batches = state.get("max_classify_batches", self.DEFAULT_MAX_BATCHES)
        selected = state.get("selected_listings", [])
        remaining = state.get("listings", [])

        if len(selected) >= quota:
            return "CONTINUE"
        if self.evals.classify_batches >= max_batches:
            return "ERROR"
        if not remaining:
            return "ERROR"
        return "NOT_ENOUGH"

    # ---------- nodes ----------

    async def create_catalog_batch(self, state: ListingSelectionState) -> dict:
        batch_id = await self.html_tools.generate_catalog_html(state["brand"])
        return {"batch_id": batch_id}

    def request_catalog(self, state: ListingSelectionState) -> dict:
        brand = state["brand"]
        batch_id = state["batch_id"]
        exists = self.html_tools.check_catalog_html(brand, batch_id)
        if not exists:
            raise RuntimeError(f"Catalog HTML not found for {brand}/{batch_id}")
        path = self.html_tools.get_catalog_html_path(brand, batch_id)
        return {"catalog_html_path": str(path)}

    def parse_catalog_html(self, state: ListingSelectionState) -> dict:
        if state.get("scrape_config"):
            self.html_tools.config = state["scrape_config"]
        listings = self.html_tools.clean_catalog_html(state["brand"], state["batch_id"])
        return {"listings": listings}

    def calculate_coverage(self, state: ListingSelectionState) -> dict:
        listings = state.get("listings", [])
        total = len(listings)
        thresholds = Listing.fields_for_coverage()
        coverage = {}
        failed = []
        for f, threshold in thresholds.items():
            count = sum(1 for l in listings if l.get(f))
            pct = count / total if total else 0
            coverage[f] = pct
            if pct < threshold:
                failed.append(f)

        self.evals.record_post_heal_coverage(coverage)

        return {"last_coverage": coverage, "failed_fields": failed}

    # Multi-field heal: full config in / full config out.
    def heal_dom_selectors(self, state: ListingSelectionState) -> dict:
        last_coverage = state.get("last_coverage", {})
        selector_history = state.get("selector_history", [])
        current_config = state.get("scrape_config") or self.html_tools.config

        self.evals.record_heal_start(last_coverage)

        goal_prompt = (
            "You are healing broken DOM selectors for a TheRealReal catalog scraper. "
            "Inspect the HTML using the available tools and return an updated "
            "scrape_config JSON with selectors that improve field coverage.\n\n"
            "Selector rules — follow these strictly:\n"
            "1. Each field's testid_suffix must be the MOST SPECIFIC suffix that "
            "uniquely identifies it. If the card exposes both `-price-original` "
            "and `-price-final`, map `price` -> `-price-original` and "
            "`final_price` -> `-price-final`. Never map two different fields to "
            "the same generic suffix like `-price`.\n"
            "2. Before assigning a suffix, call get_card_testids and prefer the "
            "longest matching suffix available. Use get_element_context to confirm "
            "the element holds the value you expect (e.g. a sale price vs an "
            "original price).\n"
            "3. Only fall back to a shorter/generic suffix (e.g. `-price`) when "
            "no more-specific variant exists in the card.\n"
            "4. Not every item is discounted. Many cards expose only a single "
            "price element (the original price) and no `-price-final`. That is "
            "expected — `final_price` is allowed to be sparse and its coverage "
            "threshold is intentionally lower. Do NOT collapse `price` and "
            "`final_price` onto the same suffix just to push final_price "
            "coverage up.\n"
            "5. Preserve fields that are already at high coverage — do not "
            "rewrite working selectors."
        )
        dynamic_context = json.dumps({
            "current_config": current_config,
            "last_coverage": last_coverage,
            "failed_fields": state.get("failed_fields", []),
            "selector_history": selector_history,
        }, indent=2)
        full_prompt = f"{goal_prompt}\n\nState:\n{dynamic_context}"

        result = self.heal_agent.invoke(
            {"messages": [HumanMessage(content=full_prompt)]},
            config={"recursion_limit": self.HEAL_RECURSION_LIMIT},
        )

        final_text = ""
        for m in reversed(result.get("messages", [])):
            if getattr(m, "type", None) != "ai":
                continue
            content = getattr(m, "content", "") or ""
            if isinstance(content, list):
                content = "".join(
                    p.get("text", "") if isinstance(p, dict) else str(p)
                    for p in content
                )
            if content.strip():
                final_text = content
                break

        if not final_text:
            print("  [heal] agent never produced final JSON; keeping old config")
            return {
                "scrape_config": current_config,
                "selector_history": list(selector_history) + [current_config],
                "retries": self.HEAL_RETRY_CAP,
            }

        cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", final_text.strip(), flags=re.MULTILINE)
        try:
            new_scrape_config = json.loads(cleaned)
        except json.JSONDecodeError:
            print(f"  [heal] non-JSON response, keeping old config. raw: {cleaned[:200]}")
            return {
                "scrape_config": current_config,
                "selector_history": list(selector_history) + [current_config],
                "retries": self.HEAL_RETRY_CAP,
            }

        return {
            "scrape_config": new_scrape_config,
            "selector_history": list(selector_history) + [current_config],
            "retries": state.get("retries", 0) + 1,
        }

    # Single-field heal: scoped input (card_testid + one field) -> scoped patch
    # merged into full config.
    def heal_single_field(self, state: ListingSelectionState) -> dict:
        last_coverage = state.get("last_coverage", {})
        selector_history = state.get("selector_history", [])
        current_config = state.get("scrape_config") or self.html_tools.config
        failed_fields = state.get("failed_fields", [])
        if not failed_fields:
            return {"retries": state.get("retries", 0) + 1}
        failed_field = failed_fields[0]

        self.evals.record_heal_start(last_coverage)

        current_fields = current_config.get("dom", {}).get("fields", {})
        prior_suffixes = [
            cfg.get("dom", {}).get("fields", {}).get(failed_field)
            for cfg in selector_history
        ]

        goal_prompt = (
            f"One field in the TheRealReal catalog scraper is broken: `{failed_field}`. "
            "Find a working testid_suffix for just that field.\n\n"
            "Workflow:\n"
            "1. Call get_element_context with a likely suffix. Prices are easy — "
            "context contains '$' or digit-comma. Try in order until one matches:\n"
            "  - for `price`: '-price-original', '-price', '-msrp'\n"
            "  - for `final_price`: '-price-final', '-price-sale'\n"
            "  - for `size`: '-size'\n"
            "2. Fallback: only call get_card_testids if step 1 exhausts candidates.\n"
            "\n"
            f"Return ONLY a JSON object with one key: {{\"{failed_field}\": \"<testid_suffix>\"}}. "
            "No fences, no prose."
        )
        dynamic_context = json.dumps({
            "card_testid": current_config.get("dom", {}).get("card_testid"),
            "fields": {failed_field: current_fields.get(failed_field)},
            "coverage": last_coverage.get(failed_field),
            "prior_attempts": prior_suffixes,
        }, indent=2)
        full_prompt = f"{goal_prompt}\n\nState:\n{dynamic_context}"

        result = self.heal_agent.invoke(
            {"messages": [HumanMessage(content=full_prompt)]},
            config={"recursion_limit": self.HEAL_RECURSION_LIMIT},
        )

        final_text = ""
        for m in reversed(result.get("messages", [])):
            if getattr(m, "type", None) != "ai":
                continue
            content = getattr(m, "content", "") or ""
            if isinstance(content, list):
                content = "".join(
                    p.get("text", "") if isinstance(p, dict) else str(p)
                    for p in content
                )
            if content.strip():
                final_text = content
                break

        if not final_text:
            print(f"  [heal-single] no final text for {failed_field}; keeping old config")
            return {
                "scrape_config": current_config,
                "selector_history": list(selector_history) + [current_config],
                "retries": self.HEAL_RETRY_CAP,
            }

        cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", final_text.strip(), flags=re.MULTILINE)
        try:
            patch = json.loads(cleaned)
        except json.JSONDecodeError:
            print(f"  [heal-single] non-JSON response: {cleaned[:200]}")
            return {
                "scrape_config": current_config,
                "selector_history": list(selector_history) + [current_config],
                "retries": self.HEAL_RETRY_CAP,
            }

        new_scrape_config = {
            **current_config,
            "dom": {
                **current_config["dom"],
                "fields": {**current_config["dom"]["fields"], **patch},
            },
        }

        return {
            "scrape_config": new_scrape_config,
            "selector_history": list(selector_history) + [current_config],
            "retries": state.get("retries", 0) + 1,
        }

    def classify_listings(self, state: ListingSelectionState) -> dict:
        listings = list(state.get("listings", []))
        selected = list(state.get("selected_listings", []))
        discarded = list(state.get("discarded_listings", []))
        confused = list(state.get("confused_listings", []))

        batch = []
        while listings and len(batch) < 3:
            candidate = listings.pop(0)
            if not candidate.get("price"):
                candidate["classification"] = {
                    "category": "confused",
                    "reason": "no price on listing — skipped without LLM call",
                }
                confused.append(candidate)
                continue
            batch.append(candidate)

        if not batch:
            return {
                "listings": listings,
                "selected_listings": selected,
                "discarded_listings": discarded,
                "confused_listings": confused,
            }

        prompt = (
            "Classify each listing as one of: 'selected', 'discarded', 'confused'. "
            "'selected' = worth researching for resale arbitrage, "
            "'discarded' = clearly not worth it, "
            "'confused' = unclear from the metadata alone. "
            "Return a JSON list of {category, reason} objects, one per listing, in order."
        )
        full_prompt = f"{prompt}\n\nListings:\n{json.dumps(batch, indent=2)}"

        chain = self.classify_llm | StrOutputParser()
        raw = chain.invoke([HumanMessage(content=full_prompt)])
        classifications = json.loads(raw)

        buckets = {"selected": 0, "discarded": 0, "confused": 0}
        for listing, cls in zip(batch, classifications):
            listing["classification"] = cls
            cat = cls.get("category", "confused")
            if cat == "selected":
                selected.append(listing)
            elif cat == "discarded":
                discarded.append(listing)
            else:
                confused.append(listing)
                cat = "confused"
            buckets[cat] += 1

        self.evals.record_classify_batch(**buckets)

        return {
            "listings": listings,
            "selected_listings": selected,
            "discarded_listings": discarded,
            "confused_listings": confused,
        }

    def save_selected_listings(self, state: ListingSelectionState) -> dict:
        return {}

    def get_listings(self) -> dict:
        return {}


class OrchestratorState(TypedDict):
    messages: str
    source_listing: dict
    sold_listings: list
    flags: list


class OrchestratorAgent:
    def __init__(self):
        pass


agent = ListingSelectionAgent("undercover")
display(Image(agent.graph.get_graph().draw_mermaid_png()))


In [ ]:
# Per-step inspection - bypasses graph entry to skip CDP fetch.
# Expects an HTML file at db/html/catalog/<brand>/*_<batch_id>.html
# Inspect `trace` (auto-displayed) or any individual step e.g. trace["parse"].

agent = ListingSelectionAgent("burberry")
state = {"brand": "burberry", "batch_id": "827s2"}
trace = {}

# manually resolve path (replaces request_catalog since we're skipping the graph)
state["catalog_html_path"] = str(
    agent.html_tools.get_catalog_html_path(state["brand"], state["batch_id"])
)

pipeline = [
    ("parse",    agent.parse_catalog_html),
    ("coverage", agent.calculate_coverage),
    ("classify", agent.classify_listings),
]
for name, node in pipeline:
    trace[name] = node(state)
    state.update(trace[name])

trace["routes"] = {
    "after_coverage": agent.route_coverage(state),
    "after_classify": agent.evaluate_picks(state),
}
trace["evals"] = vars(agent.evals)
trace["final_state"] = state
trace


In [ ]:
# Full workflow run - streams per-node updates so you can watch each step.
# create_catalog_batch is async (CDP fetch), so we use astream + top-level await.
# Requires Chrome on --remote-debugging-port=9222 + logged into TRR.

import json


def hr(title):
    print(f"\n{'=' * 8} {title} {'=' * max(0, 60 - len(title))}")


async def run_listing_selection(brand: str, quota: int = 3, max_batches: int = 5):
    agent = ListingSelectionAgent(brand)
    initial = {
        "brand": brand,
        "worthy_listing_quota": quota,
        "max_classify_batches": max_batches,
    }
    print(f"brand: {brand}  |  quota: {quota}  |  max_batches: {max_batches}")

    classify_iter = 0
    heal_iter = 0

    async for chunk in agent.graph.astream(initial, stream_mode="updates"):
        for node, update in chunk.items():
            if node == "classify_listings":
                classify_iter += 1
                hr(f"NODE: classify_listings  (iter {classify_iter})")
            elif node == "heal_dom_selectors":
                heal_iter += 1
                hr(f"NODE: heal_dom_selectors  (iter {heal_iter})")
            else:
                hr(f"NODE: {node}")

            if node == "create_catalog_batch":
                print(f"  batch_id: {update.get('batch_id')}")

            elif node == "request_catalog":
                print(f"  html path: {update.get('catalog_html_path')}")

            elif node == "parse_catalog_html":
                listings = update.get("listings", [])
                print(f"  listings parsed: {len(listings)}")
                if listings:
                    print(f"  sample[0].name: {listings[0].get('name')}")

            elif node == "calculate_coverage":
                cov = update.get("last_coverage", {})
                for f, pct in cov.items():
                    bar = "#" * int(pct * 20)
                    print(f"  {f:<14} {pct * 100:5.1f}%  {bar}")
                print(f"  failed (< {agent.COVERAGE_THRESHOLD * 100:.0f}%): {update.get('failed_fields')}")

            elif node == "heal_dom_selectors":
                print(f"  retry #{update.get('retries')}")
                print(f"  new scrape_config:")
                print(json.dumps(update.get("scrape_config"), indent=4))

            elif node == "classify_listings":
                remaining = update.get("listings", [])
                print(f"  queue remaining after batch: {len(remaining)}")
                for bucket in ("selected_listings", "discarded_listings", "confused_listings"):
                    items = update.get(bucket, [])
                    print(f"\n  {bucket} ({len(items)} total):")
                    for item in items:
                        cls = item.get("classification", {})
                        name = (item.get("name") or "?")[:60]
                        print(f"    - {name}")
                        print(f"        {cls.get('category')}: {cls.get('reason')}")

            elif node == "save_selected_listings":
                print("  (stub - persistence TODO)")

    hr("EVALS")
    for k, v in vars(agent.evals).items():
        print(f"  {k:<24} {v}")


await run_listing_selection("vivienne westwood")


In [ ]:
# Sequential HTML tool walk-through for debugging the price-coverage gap.
# Mirrors clean_catalog_html step-by-step against an existing local file so you
# can see exactly what each layer produces - no CDP fetch, no agent loop.

import re
import importlib
from pathlib import Path
import src.scraper.tools.html as _html_mod
importlib.reload(_html_mod)
HTMLTools = _html_mod.HTMLTools

BRAND = "vivienne westwood"
BATCH_ID = "fc606"  # change to whichever 062026_<id>.html you want to inspect

ht = HTMLTools(BRAND)

def hr(t): print(f"\n{'=' * 6} {t} {'=' * max(0, 60 - len(t))}")

# 1. resolve path
hr("1. resolve catalog path")
path = ht.get_catalog_html_path(BRAND, BATCH_ID)
print(f"  path: {path}")
print(f"  exists: {path is not None and Path(path).exists()}")

# 2. load soup
hr("2. load_html")
ht.load_html(str(path))
print(f"  soup loaded; html length: {len(str(ht.soup)):,}")

# 3. JSON-LD listings
hr("3. JSON-LD listings")
json_ld_cfg = ht.config["json_ld"]
raw = ht.get_json_ld_fields(json_ld_cfg["listings_path"]) or []
print(f"  listings_path: {json_ld_cfg['listings_path']}")
print(f"  raw listing count: {len(raw)}")
if raw:
    print(f"  sample[0]: {raw[0]}")

# 4. find cards
hr("4. find cards via card_testid")
card_pattern = re.compile(ht.config["dom"]["card_testid"])
cards = ht.soup.find_all("div", {"data-testid": card_pattern})
print(f"  card_testid regex: {ht.config['dom']['card_testid']}")
print(f"  cards found: {len(cards)}")

# 5. show configured DOM fields
hr("5. dom field config")
for k, v in ht.config["dom"].get("fields", {}).items():
    print(f"  {k:<16} -> suffix {v!r}")

# 6. per-card extraction with surviving-vs-missing breakdown
#    JSON-LD fields (name, listing_url, image_url) + DOM fields (price/size/etc)
hr("6. per-card field extraction (first 5 cards)")
field_map = json_ld_cfg.get("fields", {})
for i, card in enumerate(cards[:5]):
    print(f"\n  --- card {i} (testid={card.get('data-testid')}) ---")
    meta = raw[i] if i < len(raw) else {}
    for out_key, src_key in field_map.items():
        value = meta.get(src_key)
        marker = " " if value else "X"
        print(f"   {marker} {out_key:<16} json-ld[{src_key!r:<10}]    -> {value!r}")
    for cfg_field, testid_suffix in ht.config["dom"].get("fields", {}).items():
        el = card.find(attrs={"data-testid": re.compile(testid_suffix + "$")})
        value = el.get_text(strip=True) if el else None
        marker = " " if value else "X"
        print(f"   {marker} {cfg_field:<16} dom[{testid_suffix!r:<18}] -> {value!r}")

# 6b. first 5 cards that DO have a price - sanity-check what a "good" card looks like
hr("6b. first 5 cards WITH a price")
shown = 0
for i, card in enumerate(cards):
    if shown >= 5:
        break
    price_el = card.find(attrs={"data-testid": re.compile("-price-original$")})
    if not price_el:
        continue
    print(f"\n  --- card {i} (testid={card.get('data-testid')}) ---")
    meta = raw[i] if i < len(raw) else {}
    for out_key, src_key in field_map.items():
        value = meta.get(src_key)
        print(f"     {out_key:<16} json-ld[{src_key!r:<10}]    -> {value!r}")
    for cfg_field, testid_suffix in ht.config["dom"].get("fields", {}).items():
        el = card.find(attrs={"data-testid": re.compile(testid_suffix + "$")})
        value = el.get_text(strip=True) if el else None
        print(f"     {cfg_field:<16} dom[{testid_suffix!r:<18}] -> {value!r}")
    shown += 1

# 6c. indexes of all cards with vs without -price-original
hr("6c. card indexes by price presence")
with_price = [i for i, c in enumerate(cards) if c.find(attrs={"data-testid": re.compile("-price-original$")})]
without_price = [i for i, c in enumerate(cards) if not c.find(attrs={"data-testid": re.compile("-price-original$")})]
print(f"  WITH price    ({len(with_price)}): {with_price}")
print(f"  WITHOUT price ({len(without_price)}): {without_price}")

# 6d. dump testids + parent context for the first no-price card
hr("6d. anatomy of the first no-price card")
if without_price:
    idx = without_price[0]
    card = cards[idx]
    print(f"  card index: {idx}")
    print(f"  card data-testid: {card.get('data-testid')}")
    print(f"  parent tag: <{card.parent.name} data-testid={card.parent.get('data-testid')!r} class={card.parent.get('class')!r}>")
    print(f"  grandparent tag: <{card.parent.parent.name} data-testid={card.parent.parent.get('data-testid')!r} class={card.parent.parent.get('class')!r}>")
    inner_testids = [el["data-testid"] for el in card.find_all(attrs={"data-testid": True})]
    print(f"  testids inside this card ({len(inner_testids)}):")
    for t in inner_testids:
        print(f"    {t}")
    snippet = str(card)
    print(f"\n  card HTML length: {len(snippet):,}")
    print(f"  first 600 chars:\n{snippet[:600]}")
else:
    print("  (no missing-price cards to inspect)")

# 7. coverage across ALL cards for each configured field
hr("7. full-coverage per field")
total_cards = len(cards)
total_meta = len(raw)
print(f"  -- JSON-LD fields (n={total_meta}) --")
for out_key, src_key in field_map.items():
    hits = sum(1 for m in raw if m.get(src_key))
    pct = (hits / total_meta * 100) if total_meta else 0
    print(f"  {out_key:<16} json-ld[{src_key!r:<10}]    {hits}/{total_meta} ({pct:5.1f}%)")
print(f"  -- DOM fields (n={total_cards}) --")
for cfg_field, testid_suffix in ht.config["dom"].get("fields", {}).items():
    hits = sum(
        1 for c in cards
        if c.find(attrs={"data-testid": re.compile(testid_suffix + "$")})
    )
    pct = (hits / total_cards * 100) if total_cards else 0
    print(f"  {cfg_field:<16} dom[{testid_suffix!r:<18}] {hits}/{total_cards} ({pct:5.1f}%)")

# 8. agent tools - same calls the heal loop makes
hr("8. agent tool: get_card_testids (across cards)")
ids = ht.get_card_testids(card_limit=10, limit=20)
print(f"  unique testids seen: {len(ids)}")
for t in ids:
    print(f"    {t}")

hr("9. agent tool: get_element_context probes")
for suffix in ["-price-original", "-price-final", "-price", "-price-sale", "-size"]:
    ctx = ht.get_element_context(suffix, chars=60, card_limit=10, limit=5)
    print(f"\n  suffix={suffix!r}  hits={len(ctx)}")
    for c in ctx[:3]:
        print(f"    {c!r}")
